In [36]:
import numpy as np, pandas as pd
import os, json
dataset_path = "/media/lleger/LaCie/mit/disease_geometry/cxg_lung_dataset/"
import os, sys, json
sys.path.append('../../../')
from polygene.model.model import load_trained_model
from polygene.eval.metrics import prepare_cell
import numpy as np, pandas as pd, scanpy as sc
from tqdm import tqdm
import torch

neural_network_path ="/media/lleger/LaCie/mit/runs/polygene_unit_sphere/"
model, tokenizer = load_trained_model(neural_network_path, checkpoint_n=-1)

lung_cancers = ["lung adenocarcinoma", "lung large cell carcinoma", "squamous cell lung carcinoma", "small cell lung carcinoma"]
cells = [sc.read_h5ad(dataset_path + cancer + "_shard.h5ad") for cancer in lung_cancers]
cells = sc.concat([shard[:50] for shard in cells])

predictions = []
informed_predictions = []
all_labels = []
mask_id = tokenizer.token_to_id_map[tokenizer.mask_token]
for cell in tqdm(cells):
    cell_dict, labels = prepare_cell(cell, tokenizer)

    label_ids = cell_dict['input_ids'].clone()
    cell_dict['input_ids'][1:1+len(tokenizer.phenotypic_types)] = mask_id
    
    output = model(**{key: val.to(model.device).unsqueeze(0) for key, val in cell_dict.items()})

    latent_representation = output.hidden_states[:, 1+tokenizer.phenotypic_types.index('disease')].detach().cpu().numpy()
    cell_dict['input_ids'][1+len(tokenizer.phenotypic_types):] = mask_id

    # one prediction with just basic disease embedding
    cell_dict['input_ids'][1+tokenizer.phenotypic_types.index('disease')] = label_ids[1+tokenizer.phenotypic_types.index('disease')]

    output = model(**{key: val.to(model.device).unsqueeze(0) for key, val in cell_dict.items()})
    normal_prediction = output.logits.squeeze().argmax(dim=-1).clone()

    embeddings = model.embeddings(cell_dict['input_ids'].to(model.device).unsqueeze(0), cell_dict['token_type_ids'].to(model.device).unsqueeze(0))
    embeddings[:, 1+tokenizer.phenotypic_types.index('disease')] = torch.tensor(latent_representation, device=model.device, dtype=torch.float32)

    cell_dict['inputs_embeds'] = embeddings.squeeze()
    output = model(**{key: val.to(model.device).unsqueeze(0) for key, val in cell_dict.items()})
    uncertainty_informed_prediction = output.logits.squeeze().argmax(dim=-1).clone()
    predictions.append(normal_prediction.detach().cpu().numpy()[1 + len(tokenizer.phenotypic_types):])
    informed_predictions.append(uncertainty_informed_prediction.detach().cpu().numpy()[1 + len(tokenizer.phenotypic_types):])
    all_labels.append(label_ids.detach().cpu().numpy()[1 + len(tokenizer.phenotypic_types):])

loading checkpoint-1500000


100%|██████████| 200/200 [00:05<00:00, 36.82it/s]


In [38]:
labels = np.concatenate(all_labels)
preds = np.concatenate(predictions)
informed = np.concatenate(informed_predictions)

# where does informed help vs hurt?
improved = (informed == labels) & (preds != labels)
degraded = (informed != labels) & (preds == labels)

print(f"improved: {improved.sum()}, degraded: {degraded.sum()}")

improved: 9841, degraded: 7456


In [40]:
pd.Series(labels).value_counts()

1099    96308
1098    70020
1100    41510
1101    13218
1102     3688
1103      478
1104      108
1105       20
1106        2
dtype: int64

In [39]:
labels = np.concatenate(all_labels)
preds = np.concatenate(predictions)
informed = np.concatenate(informed_predictions)

improved = (informed == labels) & (preds != labels)
degraded = (informed != labels) & (preds == labels)

print("improved on:")
print(pd.Series(labels[improved]).value_counts())

print("\ndegraded on:")
print(pd.Series(labels[degraded]).value_counts())

improved on:
1098    4832
1099    4423
1100     350
1101     121
1102     105
1104       5
1103       5
dtype: int64

degraded on:
1100    3291
1099    2576
1101    1512
1102      36
1098      24
1103      12
1104       4
1105       1
dtype: int64


In [41]:
from sklearn.metrics import balanced_accuracy_score, f1_score

labels = np.concatenate(all_labels)
preds = np.concatenate(predictions)
informed = np.concatenate(informed_predictions)

print("overall accuracy:", (labels == preds).mean(), (labels == informed).mean())
print("balanced accuracy:", balanced_accuracy_score(labels, preds), balanced_accuracy_score(labels, informed))
print("macro f1:", f1_score(labels, preds, average='macro'), f1_score(labels, informed, average='macro'))

overall accuracy: 0.7554004402002201 0.7659838829919415
balanced accuracy: 0.6374284033823285 0.6235487035798256
macro f1: 0.6623343741125101 0.6594901331763068


In [37]:
#pd.DataFrame([np.concatenate(all_labels)])
print((np.concatenate(all_labels) == np.concatenate(predictions)).mean(),
(np.concatenate(all_labels) == np.concatenate(informed_predictions)).mean())


0.7554004402002201 0.7659838829919415
